# Stage 0 — Freeze the dated FDA snapshot

**Run this once.** It pins the entire study to a single date so every downstream number is
reproducible forever, regardless of how FDA's live database grows afterward.

It does three things:
1. Derives the **154 product codes** from an explicit inclusion rule (D2) — orthopedic advisory
   panel + device class II + implant flag + regulation 888.3xxx — so the corpus boundary is a
   stated rule, not a hand-picked list.
2. Downloads every 510(k), recall, and enforcement record for those codes **once**.
3. Writes the raw responses to `snapshot/` plus `SNAPSHOT.json` (date, counts, openFDA release date).

Network: yes — this is the only stage that must run on the snapshot date. After it, every other
stage reads these frozen files.

In [ ]:
import urllib.request, urllib.parse, json, gzip, time, datetime, os
import pandas as pd

def fda_get(endpoint, params):
    url = f"https://api.fda.gov/device/{endpoint}.json?" + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url, timeout=90) as r:
        return json.loads(r.read())

def fetch_all(endpoint, search):
    """Page through every record matching `search`."""
    out, skip = [], 0
    while True:
        try:
            j = fda_get(endpoint, {"search": search, "limit": 1000, "skip": skip})
        except urllib.error.HTTPError as e:
            if e.code == 404:
                break
            raise
        res = j.get("results", [])
        if not res:
            break
        out += res
        skip += 1000
        if skip >= j["meta"]["results"]["total"]:
            break
        time.sleep(0.05)
    return out

### D2 inclusion rule — derive the 154 product codes

In [ ]:
cls_rows, skip = [], 0
while True:
    j = fda_get("classification", {"search": 'medical_specialty_description:"Orthopedic"',
                                    "limit": 1000, "skip": skip})
    res = j.get("results", [])
    if not res:
        break
    cls_rows += res
    skip += 1000
    if skip >= j["meta"]["results"]["total"]:
        break
cls = pd.DataFrame(cls_rows)
rule = ((cls["device_class"] == "2")
        & (cls["implant_flag"].astype(str).str.upper().str.startswith("Y"))
        & (cls["regulation_number"].astype(str).str.startswith("888.3")))
CODES = sorted(cls.loc[rule, "product_code"].unique().tolist())
print(f"inclusion rule -> {len(CODES)} product codes")

### Freeze 510(k), recall, and enforcement records

In [ ]:
os.makedirs("snapshot", exist_ok=True)
SNAP_DATE = datetime.date.today().isoformat()

k510 = []
for c in CODES:
    k510 += fetch_all("510k", f"product_code:{c}")
print(f"510(k) raw records: {len(k510)}")

recalls = []
for c in CODES:
    recalls += fetch_all("recall", f"product_code:{c}")
prns = sorted({r.get("product_res_number") for r in recalls if r.get("product_res_number")})
print(f"recall raw records: {len(recalls)} | unique recall numbers: {len(prns)}")

enforcement = []
for prn in prns:
    try:
        res = fda_get("enforcement", {"search": f'recall_number:"{prn}"', "limit": 1}).get("results", [])
        if res:
            enforcement.append(res[0])
    except Exception:
        pass
print(f"enforcement raw records: {len(enforcement)}")

### Write the frozen files + manifest

In [ ]:
with gzip.open("snapshot/510k_raw.json.gz", "wt") as f:
    json.dump(k510, f)
with gzip.open("snapshot/recall_raw.json.gz", "wt") as f:
    json.dump(recalls, f)
with gzip.open("snapshot/enforcement_raw.json.gz", "wt") as f:
    json.dump(enforcement, f)

release = fda_get("510k", {"limit": 1})["meta"].get("last_updated")
uniq = pd.DataFrame(k510).drop_duplicates(subset=["k_number"])
manifest = {
    "snapshot_date": SNAP_DATE,
    "openfda_last_updated": release,
    "inclusion_rule": "Orthopedic panel + Class II + implant_flag=Y + regulation 888.3xxx",
    "n_codes": len(CODES),
    "product_codes": CODES,
    "n_510k_raw": len(k510),
    "n_unique_devices": int(uniq["k_number"].nunique()),
    "n_with_summary": int((uniq["statement_or_summary"] == "Summary").sum()),
    "n_recall_raw": len(recalls),
    "n_enforcement_raw": len(enforcement),
}
json.dump(manifest, open("snapshot/SNAPSHOT.json", "w"), indent=2)

print("CHECKPOINT  snapshot written")
print(f"CHECKPOINT  date {SNAP_DATE} | codes {len(CODES)} | devices {manifest['n_unique_devices']} "
      f"| summaries {manifest['n_with_summary']} | recalls {len(recalls)}")